In [1]:
"""
08_random_forest.py
"""

'\n08_random_forest.py\n'

In [2]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

df = pd.read_csv(str(project_root / "data" / "processed" / "model_features.csv"))

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from utils.print_section import print_section

print(df.shape)

# ============================================================
# Features
# ============================================================

FEATURES = [
    "profitability",
    "liquidity",
    "solvency",
    "structure",
    "log_age",
    "size",
]

TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# ============================================================
# Scale positive class
# ============================================================

negative_class = (y_train == 0).sum()
positive_class = (y_train == 1).sum()

scale_pos_weight = (
    negative_class / positive_class
)

print_section("Class imbalance")

print(f"Negative class: {negative_class:,}")
print(f"Positive class: {positive_class:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# ============================================================
# Model
# ============================================================

pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            eval_metric="logloss",
        )
    )
])

# ============================================================
# Fit
# ============================================================

pipeline.fit(
    X_train,
    y_train,
)

# ============================================================
# Predict
# ============================================================

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

# ============================================================
# Performance
# ============================================================

print_section("Performance")

print(
    f"Accuracy : {accuracy_score(y_test, y_pred):.4f}"
)

print(
    f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}"
)

print(
    f"Recall   : {recall_score(y_test, y_pred):.4f}"
)

print(
    f"F1-score : {f1_score(y_test, y_pred):.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}"
)

# ============================================================
# Confusion matrix
# ============================================================

print_section("Confusion matrix")

cm = confusion_matrix(
    y_test,
    y_pred,
)

print(cm)

tn, fp, fn, tp = cm.ravel()

print()

print(f"True Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

# ============================================================
# Classification report
# ============================================================

print_section("Classification report")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0,
    )
)

# ============================================================
# Feature importance
# ============================================================

print_section("Feature importance")

importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": (
        pipeline
        .named_steps["model"]
        .feature_importances_
    )
})

importances = importances.sort_values(
    by="importance",
    ascending=False,
)

print(importances)

# ============================================================
# Highest predicted probabilities
# ============================================================

print_section("Highest predicted probabilities")

prob_df = pd.DataFrame({
    "actual": y_test.values,
    "probability": y_prob,
})

print(
    prob_df
    .sort_values(
        by="probability",
        ascending=False,
    )
    .head(20)
)

# ============================================================
# Baseline
# ============================================================

print_section("Baseline")

print(
    f"Failure rate: {y.mean():.4%}"
)

# ============================================================
# Model improvement over baseline
# ============================================================

print_section("Model improvement over baseline")

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0,
)

print(
    f"Baseline failure rate: "
    f"{y.mean():.4%}"
)

print(
    f"Model precision: "
    f"{precision:.4%}"
)

print(
    f"Improvement factor: "
    f"{precision / y.mean():.2f}x"
)

(981818, 96)

Class imbalance
Negative class: 783,196
Positive class: 2,258
scale_pos_weight: 346.85

Performance
Accuracy : 0.8116
Precision: 0.0107
Recall   : 0.7074
F1-score : 0.0211
ROC-AUC  : 0.8438

Confusion matrix
[[158973  36827]
 [   165    399]]

True Negatives : 158,973
False Positives: 36,827
False Negatives: 165
True Positives : 399

Classification report
              precision    recall  f1-score   support

           0       1.00      0.81      0.90    195800
           1       0.01      0.71      0.02       564

    accuracy                           0.81    196364
   macro avg       0.50      0.76      0.46    196364
weighted avg       1.00      0.81      0.89    196364


Feature importance
         feature  importance
2       solvency    0.284912
0  profitability    0.232697
3      structure    0.143388
1      liquidity    0.138362
5           size    0.120131
4        log_age    0.080511

Highest predicted probabilities
        actual  probability
37298        1   